In [239]:
import numpy as np
import pandas as pd
from numpy import nan

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

In [240]:
dev = pd.read_csv("winter_project_2026/development.csv")
eva= pd.read_csv("winter_project_2026/evaluation.csv")

In [241]:
display(dev)
print(dev.columns)

columns = dev.columns
for col in columns:
    print(f"{col} :{dev.iloc[0][col]}")

,Id,source,title,article,page_rank,timestamp,label
0,0,AllAfrica.com,OPEC Boosts Nigeria&#39;s Oil Revenue By .82m Bpd,THE Organisation of Petroleum Exporting Countr...,5,2004-09-16 22:39:53,5
1,1,Xinhua,Yearender: Mideast peace roadmap reaches dead-...,Looking back at the major events that took pla...,5,2004-12-17 19:01:14,0
2,2,Yahoo,Battleground Dispatches for Oct. 5 \\n (CQP...,CQPolitics.com - Here are today's Battleground...,5,2006-10-05 18:42:29,0
3,3,BBC,Air best to resuscitate newborns,Air rather than oxygen should be used to resus...,5,0000-00-00 00:00:00,0
4,4,Yahoo,High tech German train crash kills at least on...,"<p><a href=""http://us.rd.yahoo.com/dailynews/r...",5,2006-09-22 17:28:57,0
...,...,...,...,...,...,...,...
79992,79992,Yahoo,Italy's embattled Prodi faces vote of confiden...,"<p><a href=""http://us.rd.yahoo.com/dailynews/r...",5,2008-01-23 11:39:35,0
79993,79993,All-Baseball.com,"Ding Dong, the Deal is Dead","As yesterday began, there was widespread antic...",5,0000-00-00 00:00:00,4
79994,79994,Yahoo,Two bombs discovered in Sardinia after Berlusc...,AFP - Police discovered two bombs near the Sar...,5,0000-00-00 00:00:00,0
79995,79995,Voice,Red Cross Report Alleges US Detainee Abuse at ...,A report by the International Committee of the...,5,0000-00-00 00:00:00,3


Index(['Id', 'source', 'title', 'article', 'page_rank', 'timestamp', 'label'], dtype='object')
Id :0
source :AllAfrica.com
title :OPEC Boosts Nigeria&#39;s Oil Revenue By .82m Bpd
article :THE Organisation of Petroleum Exporting Countries (OPEC) is hiking its official output by one million barrels per day effective November with Nigeria getting 82,000 barrels per day or 8.2 per cent of the new quota.
page_rank :5
timestamp :2004-09-16 22:39:53
label :5


In [242]:
print(f"shape dev:{dev.shape}")
print(f"shape eva:{eva.shape}")
for col in columns:
    print(f"unique {col}: {len(dev[col].unique())}")

shape dev:(79997, 7)
shape eva:(20000, 6)
unique Id: 79997
unique source: 1359
unique title: 75784
unique article: 74394
unique page_rank: 4
unique timestamp: 52183
unique label: 7


In [243]:
#pd.isna(dev).sum()
rows = dev.loc[dev.duplicated(subset=['article', 'title'])]
print(dev.duplicated(subset=['article']).sum())
print(dev.duplicated(subset=['title']).sum())
print(dev.duplicated(subset=['title','article']).sum())
duplicated = dev.loc[dev.duplicated(subset=['title','article'])]
#print(dev.duplicated(subset=['timestamp']).sum())
mask = dev['article'].isin(duplicated['article'])
to_drop = dev[mask]
dev.drop(to_drop["Id"], inplace=True)
dev.drop("Id", inplace=True, axis=1)
dev.dropna(inplace=True)

5603
4213
2954


In [244]:
dev

,source,title,article,page_rank,timestamp,label
0,AllAfrica.com,OPEC Boosts Nigeria&#39;s Oil Revenue By .82m Bpd,THE Organisation of Petroleum Exporting Countr...,5,2004-09-16 22:39:53,5
1,Xinhua,Yearender: Mideast peace roadmap reaches dead-...,Looking back at the major events that took pla...,5,2004-12-17 19:01:14,0
2,Yahoo,Battleground Dispatches for Oct. 5 \\n (CQP...,CQPolitics.com - Here are today's Battleground...,5,2006-10-05 18:42:29,0
3,BBC,Air best to resuscitate newborns,Air rather than oxygen should be used to resus...,5,0000-00-00 00:00:00,0
4,Yahoo,High tech German train crash kills at least on...,"<p><a href=""http://us.rd.yahoo.com/dailynews/r...",5,2006-09-22 17:28:57,0
...,...,...,...,...,...,...
79991,RedNova,LEDs Move Into Home Lighting Market,"By MARK JEWELL EVERETT, Mass. - Joey Nicotera...",4,2007-06-25 07:08:21,2
79992,Yahoo,Italy's embattled Prodi faces vote of confiden...,"<p><a href=""http://us.rd.yahoo.com/dailynews/r...",5,2008-01-23 11:39:35,0
79994,Yahoo,Two bombs discovered in Sardinia after Berlusc...,AFP - Police discovered two bombs near the Sar...,5,0000-00-00 00:00:00,0
79995,Voice,Red Cross Report Alleges US Detainee Abuse at ...,A report by the International Committee of the...,5,0000-00-00 00:00:00,3


In [245]:
from sklearn.feature_extraction.text import TfidfVectorizer

art = dev['article']
title = dev['title']

vectorizer = TfidfVectorizer(stop_words='english',
                             max_features=10000, 
                             max_df=0.6,
                             min_df=5)


X_art = vectorizer.fit_transform(art)
A_art = X_art.toarray()


In [246]:
def rSVD(X, k, p=10, q=2):
    # X m,n
    # Om n, k+p
    # Y m, k+p 
    m, n = X.shape
    Omega = np.random.normal(size=(n, k + p)) # P = oversampling, increase drastically the aìprobability of extracting usefull data
    Y = X @ Omega
    for _ in range(q):   #POWER ITERATIONS, increase big eigenvals, decrease small eigenvals
        Y = X @ (X.T @ Y)
    Q, _ = np.linalg.qr(Y)
    # Q m, k+p
    B = Q.T @ X   # B is the projection of X in the smaller space ->  dim :(k + p ), n
    # B k+p, n
    U_hat, Sigma, Vt = np.linalg.svd(B, full_matrices=False)
    # Sigma k+p, k+p
    # U_hat k+p, k+p
    # Vt k+p, n
    U = Q @ U_hat # Back to the bigger space
    # U m, k+p
    return U[:, :k], Sigma[:k], Vt[:k, :]       #We return without oversampling col/rows  
    # U m, k
    # Sigma k, k
    # Vt k, n

In [247]:
U_art, Sigma_art, Vt_art = rSVD(A_art, 100)

/var/folders/gw/h_3h_b5d74v_ds3nx7nhw4hh0000gn/T/ipykernel_13031/3221177329.py:7: RuntimeWarning: divide by zero encountered in matmul
  Y = X @ Omega
/var/folders/gw/h_3h_b5d74v_ds3nx7nhw4hh0000gn/T/ipykernel_13031/3221177329.py:7: RuntimeWarning: overflow encountered in matmul
  Y = X @ Omega
/var/folders/gw/h_3h_b5d74v_ds3nx7nhw4hh0000gn/T/ipykernel_13031/3221177329.py:7: RuntimeWarning: invalid value encountered in matmul
  Y = X @ Omega
/var/folders/gw/h_3h_b5d74v_ds3nx7nhw4hh0000gn/T/ipykernel_13031/3221177329.py:9: RuntimeWarning: divide by zero encountered in matmul
  Y = X @ (X.T @ Y)
/var/folders/gw/h_3h_b5d74v_ds3nx7nhw4hh0000gn/T/ipykernel_13031/3221177329.py:9: RuntimeWarning: overflow encountered in matmul
  Y = X @ (X.T @ Y)
/var/folders/gw/h_3h_b5d74v_ds3nx7nhw4hh0000gn/T/ipykernel_13031/3221177329.py:9: RuntimeWarning: invalid value encountered in matmul
  Y = X @ (X.T @ Y)
/var/folders/gw/h_3h_b5d74v_ds3nx7nhw4hh0000gn/T/ipykernel_13031/3221177329.py:12: RuntimeWarnin

In [248]:
X_title = vectorizer.fit_transform(title)
A_title = X_title.toarray()

U_title, Sigma_title, Vt_title = rSVD(A_title, 50)


/var/folders/gw/h_3h_b5d74v_ds3nx7nhw4hh0000gn/T/ipykernel_13031/3221177329.py:7: RuntimeWarning: divide by zero encountered in matmul
  Y = X @ Omega
/var/folders/gw/h_3h_b5d74v_ds3nx7nhw4hh0000gn/T/ipykernel_13031/3221177329.py:7: RuntimeWarning: overflow encountered in matmul
  Y = X @ Omega
/var/folders/gw/h_3h_b5d74v_ds3nx7nhw4hh0000gn/T/ipykernel_13031/3221177329.py:7: RuntimeWarning: invalid value encountered in matmul
  Y = X @ Omega
/var/folders/gw/h_3h_b5d74v_ds3nx7nhw4hh0000gn/T/ipykernel_13031/3221177329.py:9: RuntimeWarning: divide by zero encountered in matmul
  Y = X @ (X.T @ Y)
/var/folders/gw/h_3h_b5d74v_ds3nx7nhw4hh0000gn/T/ipykernel_13031/3221177329.py:9: RuntimeWarning: overflow encountered in matmul
  Y = X @ (X.T @ Y)
/var/folders/gw/h_3h_b5d74v_ds3nx7nhw4hh0000gn/T/ipykernel_13031/3221177329.py:9: RuntimeWarning: invalid value encountered in matmul
  Y = X @ (X.T @ Y)
/var/folders/gw/h_3h_b5d74v_ds3nx7nhw4hh0000gn/T/ipykernel_13031/3221177329.py:12: RuntimeWarnin

In [255]:
print(U_art.shape)
print(U_title.shape)
print(dev.shape)

(72585, 100)
(72585, 50)
(72585, 6)


In [263]:
df_title = pd.DataFrame(U_title)
df_art = pd.DataFrame(U_art)
print(df_art.shape , df_title.shape, dev.shape)
df = pd.DataFrame([dev, df_title, df_art])
df


(72585, 100) (72585, 50) (72585, 6)


ValueError: setting an array element with a sequence. The requested array has an inhomogeneous shape after 2 dimensions. The detected shape was (3, 72585) + inhomogeneous part.